# Notebook 00: Setup + API Warmup (DCC26)

Reference walkthrough for inspecting EngiBench as a reproducible benchmark contract.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

Use this notebook as the baseline interpretation layer before any model training.
The objective is conceptual correctness, not speed.


## Why this warmup matters

Most benchmarking disagreements come from interface misunderstandings, not algorithmic novelty.
This chapter removes that ambiguity up front.


## Optional install cell (fresh Colab)

Only required on fresh or reset Colab runtimes.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'matplotlib', 'seaborn']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


### Step 1 - Initialize reproducible session

Seed and version information define your execution context.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import engibench
from engibench.problems.beams2d.v0 import Beams2D

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

print('engibench version:', engibench.__version__)
print('seed:', SEED)

### Step 2 - Instantiate benchmark problem

Inspect problem metadata and ensure the design/condition/objective contract is explicit.


In [ ]:
problem = Beams2D(seed=SEED)

print('Problem class:', type(problem).__name__)
print('Design space:', problem.design_space)
print('Objectives:', problem.objectives)
print('Conditions instance:', problem.conditions)
print('Condition keys:', problem.conditions_keys)
print('Dataset ID:', problem.dataset_id)

### Step 3 - Inspect dataset structure

Confirm split semantics and field consistency before using this data in modeling notebooks.


In [ ]:
dataset = problem.dataset
print(dataset)

sample_idx = 0
design = np.array(dataset['train']['optimal_design'][sample_idx])
config = {k: dataset['train'][k][sample_idx] for k in problem.conditions_keys}

print('Sample design shape:', design.shape)
print('Sample config:', config)

### Step 4 - Visualize one benchmark design

Use rendering to align numerical representation with engineering intuition.


In [ ]:
fig, ax = problem.render(design)
ax.set_title('Sample Beams2D design from training split')
plt.show()

### Step 5 - Test constraint semantics

A controlled violation case clarifies what `check_constraints` is actually diagnosing.


In [ ]:
# One explicit constraint check with intentionally mismatched volume fraction
bad_config = dict(config)
bad_config['volfrac'] = 0.2
violations = problem.check_constraints(design=design, config=bad_config)

print('Violation count:', len(violations))
if violations:
    print(violations)
else:
    print('No violations found')

## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Next

Continue with Notebook 01 for model integration and artifact generation.


## Reflection prompts

- What would make two reported results incomparable on this benchmark?
- Which configuration fields are non-negotiable in method reporting?


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
